<a href="https://colab.research.google.com/github/kimheeseo/LSCNS/blob/main/Copper_Tube_Diameter_Prediction_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
동관 내경 추정 계산기
====================
충격에너지(E = 0.5 * m * v²) 기반으로
DA 및 HA 동관 내경을 추정하는 Power-Law 회귀식 적용

[데이터 출처 - 표]
Risk Factor    | Energy | DA 동관 내경 | HA 동관 내경
40kg, 1.4m/s  |  39 J  |  4.81 mm    |  5.45 mm
60kg, 1.7m/s  |  87 J  |  3.76 mm    |  4.48 mm
65kg, 1.8m/s  | 100 J  |  3.46 mm    |  4.18 mm
70kg, 1.9m/s  | 126 J  |  3.15 mm    |  3.87 mm
80kg, 2.0m/s  | 160 J  |  2.67 mm    |  3.45 mm
"""

import math
from scipy.optimize import curve_fit
import numpy as np

# ─────────────────────────────────────────────
# 1. 학습 데이터
# ─────────────────────────────────────────────
ENERGY = np.array([39, 87, 100, 126, 160])   # [J]
DA_OBS = np.array([4.81, 3.76, 3.46, 3.15, 2.67])  # [mm]
HA_OBS = np.array([5.45, 4.48, 4.18, 3.87, 3.45])  # [mm]

# ─────────────────────────────────────────────
# 2. Power-Law 회귀 피팅
# ─────────────────────────────────────────────
def power_law(E, a, b):
    return a * np.power(E, b)

(aDA, bDA), _ = curve_fit(power_law, ENERGY, DA_OBS, p0=[20, -0.4])
(aHA, bHA), _ = curve_fit(power_law, ENERGY, HA_OBS, p0=[20, -0.4])

# R² 계산
def r_squared(obs, pred):
    ss_res = np.sum((obs - pred) ** 2)
    ss_tot = np.sum((obs - np.mean(obs)) ** 2)
    return 1 - ss_res / ss_tot

r2_DA = r_squared(DA_OBS, power_law(ENERGY, aDA, bDA))
r2_HA = r_squared(HA_OBS, power_law(ENERGY, aHA, bHA))

# m, v 관계식 계수 (E = 0.5*m*v² 대입)
# d = a * (0.5*m*v²)^b = (a * 0.5^b) * m^b * v^(2b)
kDA = aDA * (0.5 ** bDA)
kHA = aHA * (0.5 ** bHA)

# ─────────────────────────────────────────────
# 3. 추정 함수
# ─────────────────────────────────────────────
def estimate_from_energy(energy_J: float) -> tuple[float, float]:
    """
    충격에너지(J)로부터 DA, HA 동관 내경(mm) 추정

    Parameters
    ----------
    energy_J : float  충격에너지 [J]

    Returns
    -------
    (da_mm, ha_mm) : tuple[float, float]
    """
    da = aDA * (energy_J ** bDA)
    ha = aHA * (energy_J ** bHA)
    return da, ha


def estimate_from_mv(mass_kg: float, velocity_ms: float) -> tuple[float, float]:
    """
    질량(kg)과 속도(m/s)로부터 DA, HA 동관 내경(mm) 추정

    Parameters
    ----------
    mass_kg     : float  앵커 질량 [kg]
    velocity_ms : float  충돌 속도 [m/s]

    Returns
    -------
    (da_mm, ha_mm) : tuple[float, float]
    """
    # E = 0.5 * m * v²  [J]
    energy_J = 0.5 * mass_kg * (velocity_ms ** 2)
    return estimate_from_energy(energy_J)


# ─────────────────────────────────────────────
# 4. 결과 출력
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 65)
    print(" 동관 내경 추정 회귀식 (Power-Law Fitting)")
    print("=" * 65)
    print()
    print("▶ 에너지 기반 관계식")
    print(f"   DA 내경 [mm] = {aDA:.4f} × E^({bDA:.4f})    R² = {r2_DA:.4f}")
    print(f"   HA 내경 [mm] = {aHA:.4f} × E^({bHA:.4f})    R² = {r2_HA:.4f}")
    print()
    print("▶ 질량·속도 기반 관계식  (E = ½mv² 대입)")
    print(f"   DA 내경 [mm] = {kDA:.4f} × m^({bDA:.4f}) × v^({2*bDA:.4f})")
    print(f"   HA 내경 [mm] = {kHA:.4f} × m^({bHA:.4f}) × v^({2*bHA:.4f})")
    print()

    print("-" * 65)
    print(" 피팅 검증 (원본 데이터 vs 추정값)")
    print("-" * 65)
    print(f"{'Risk Factor':20s} {'E[J]':>6} {'DA실제':>8} {'DA추정':>8} {'HA실제':>8} {'HA추정':>8}")
    labels = ["40kg,1.4m/s","60kg,1.7m/s","65kg,1.8m/s","70kg,1.9m/s","80kg,2.0m/s"]
    for i, label in enumerate(labels):
        da_pred, ha_pred = estimate_from_energy(ENERGY[i])
        print(f"{label:20s} {ENERGY[i]:>6.0f} {DA_OBS[i]:>8.2f} {da_pred:>8.2f} {HA_OBS[i]:>8.2f} {ha_pred:>8.2f}")

    print()
    print("=" * 65)
    print(" 사용 예시")
    print("=" * 65)

    examples = [(40, 1.4), (70, 1.9), (100, 2.5), (50, 1.6)]
    for m, v in examples:
        E  = 0.5 * m * v**2
        da, ha = estimate_from_mv(m, v)
        print(f"  {m}kg, {v}m/s  →  E = {E:.1f}J  |  DA = {da:.2f} mm  |  HA = {ha:.2f} mm")

    print()
    print("▶ 함수 직접 호출 방법:")
    print("   estimate_from_mv(mass_kg, velocity_ms)   → (da_mm, ha_mm)")
    print("   estimate_from_energy(energy_J)            → (da_mm, ha_mm)")

 동관 내경 추정 회귀식 (Power-Law Fitting)

▶ 에너지 기반 관계식
   DA 내경 [mm] = 19.6229 × E^(-0.3800)    R² = 0.9735
   HA 내경 [mm] = 16.7560 × E^(-0.3034)    R² = 0.9777

▶ 질량·속도 기반 관계식  (E = ½mv² 대입)
   DA 내경 [mm] = 25.5362 × m^(-0.3800) × v^(-0.7600)
   HA 내경 [mm] = 20.6779 × m^(-0.3034) × v^(-0.6068)

-----------------------------------------------------------------
 피팅 검증 (원본 데이터 vs 추정값)
-----------------------------------------------------------------
Risk Factor            E[J]     DA실제     DA추정     HA실제     HA추정
40kg,1.4m/s              39     4.81     4.88     5.45     5.51
60kg,1.7m/s              87     3.76     3.60     4.48     4.32
65kg,1.8m/s             100     3.46     3.41     4.18     4.14
70kg,1.9m/s             126     3.15     3.12     3.87     3.86
80kg,2.0m/s             160     2.67     2.85     3.45     3.59

 사용 예시
  40kg, 1.4m/s  →  E = 39.2J  |  DA = 4.87 mm  |  HA = 5.50 mm
  70kg, 1.9m/s  →  E = 126.3J  |  DA = 3.12 mm  |  HA = 3.86 mm
  100kg, 2.5m/s  →  E = 312.5J  |  D